# Vì sao chốt `floor20_open2`? — Lưới 60 cấu hình × 693 cặp × 2 điều kiện

Hai tham số quyết định "nhạy" hay "sạch" của bản đồ lỗi:

- `threshold_floor` — sàn của ngưỡng Otsu: `ngưỡng = max(Otsu, floor)`.
  Floor thấp → giữ được vết lỗi mờ (tăng recall) nhưng nhiều đốm giả.
- `open_ksize` — kernel morphology opening: xoá nhiễu nhỏ, nhưng xoá luôn lỗi vài pixel.

Lưới đã chạy: `floor ∈ {10,15,20,25,30}` × `open ∈ {1,2,3}` × `hysteresis ∈ {0, 0.5}` ×
`min_area ∈ {10,20}` = **60 cấu hình**, mỗi cấu hình chạy **2 điều kiện**:

- `clean` — ảnh gốc.
- `resample` — nắn lệch giả lập rồi căn lại (mô phỏng camera lệch).

**Nguyên tắc**

- Số chính thức đọc từ `experiments/detection_params_summary.csv` (120 dòng).
- Notebook chỉ đọc `data/`, không ghi đè CSV; hình ghi ra thư mục tạm.
- ⚠️ Cột `is_baseline=True` trong CSV đánh dấu `floor30_open3` — đó là cấu hình **gốc lúc chạy lưới**.
  Từ 17/11 nhóm đổi default sang `floor20_open2`, nên `DEFAULT_*` trong mã và cột `is_baseline` trong CSV
  không còn trùng nhau. Điều này có chủ đích, không phải lỗi.


In [ ]:

from pathlib import Path
import csv
import os
import sys
import tempfile
import time

# Tìm project root (thư mục chứa dataset_utils.py) để notebook chạy được từ mọi nơi.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "dataset_utils.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "experiments"))

import cv2
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

%matplotlib inline

CHART_DIR = Path(tempfile.gettempdir()) / "pcb_notebook_charts"
CHART_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

import argparse

from dataset_utils import find_pairs, load_image
from evaluation import detection_counts, read_labels
from module_visualization import draw_results
import exp_detection_params as edp

print("Project root :", PROJECT_ROOT)
print("BASELINE theo DEFAULT_* hiện tại:", edp.BASELINE, "->", edp.config_name(edp.BASELINE))


def to_display(image, max_width: int = 1000):
    """BGR/gray -> RGB và thu nhỏ để hiển thị."""
    if image.ndim == 2:
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
    else:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    height, width = image.shape[:2]
    if width > max_width:
        scale = max_width / width
        image = cv2.resize(image, (max_width, int(height * scale)), interpolation=cv2.INTER_AREA)
    return image


def show_stack(items, figsize=(14, 4.2)):
    fig, axes = plt.subplots(len(items), 1, figsize=(figsize[0], figsize[1] * len(items)),
                             squeeze=False)
    for row, (image, title) in enumerate(items):
        axes[row, 0].imshow(to_display(image))
        axes[row, 0].set_title(title, fontsize=10)
        axes[row, 0].axis("off")
    plt.tight_layout()
    plt.show()


## 1. Đọc lưới tham số

Mỗi dòng = một (điều kiện, cấu hình). Các cột quan trọng: `recall`, `precision`, `f1`,
`false_blobs_per_img`, `pairs_all_missed`, `recall_<lớp>`, `kpi_pass`.


In [ ]:
summary_csv = PROJECT_ROOT / "experiments" / "detection_params_summary.csv"
with summary_csv.open("r", encoding="utf-8-sig", newline="") as handle:
    grid = list(csv.DictReader(handle))

by_cfg = {(row["condition"], row["config"]): row for row in grid}
configs = sorted({row["config"] for row in grid})
conditions = sorted({row["condition"] for row in grid})

print(f"{len(grid)} dòng = {len(configs)} cấu hình × {len(conditions)} điều kiện")
print("floor      :", sorted({row['floor'] for row in grid}, key=int))
print("open       :", sorted({row['open'] for row in grid}, key=int))
print("hysteresis :", sorted({row['hysteresis'] for row in grid}, key=float))
print("min_area   :", sorted({row['min_area'] for row in grid}, key=int))


## 2. Hai cấu hình cần so: gốc và cấu hình nhóm chốt


In [ ]:
OLD = "floor30_open3_single_area20"   # cấu hình gốc lúc chạy lưới
NEW = "floor20_open2_single_area20"   # nhóm chốt 17/11

def show_config(name):
    for condition in ("clean", "resample"):
        row = by_cfg[(condition, name)]
        print(f"  {condition:<9} R {float(row['recall']):.5f} | P {float(row['precision']):.5f} | "
              f"F1 {float(row['f1']):.5f} | đốm giả/ảnh {float(row['false_blobs_per_img']):.3f} | "
              f"cặp sót toàn bộ {row['pairs_all_missed']}")

print("Gốc (floor30/open3):")
show_config(OLD)
print("Chốt (floor20/open2):")
show_config(NEW)

print("\nChênh lệch (mới − cũ):")
print(f"{'chỉ số':<24}{'clean':>12}{'resample':>12}")
for field in ("recall", "precision", "f1", "false_blobs_per_img"):
    delta = [float(by_cfg[(c, NEW)][field]) - float(by_cfg[(c, OLD)][field]) for c in ("clean", "resample")]
    print(f"{field:<24}{delta[0]:>+12.5f}{delta[1]:>+12.5f}")


## 3. Đánh đổi recall ↔ báo động giả

Mỗi điểm là một cấu hình. Đường đứt là KPI nhóm đặt ra. Vùng "đạt KPI" là góc trên-trái —
và **không điểm nào nằm trong đó** (xem mục 6).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2), sharey=True)
for ax, condition in zip(axes, ("clean", "resample")):
    rows = [r for r in grid if r["condition"] == condition]
    ax.scatter([float(r["false_blobs_per_img"]) for r in rows],
               [float(r["recall"]) for r in rows],
               s=28, alpha=0.55, color="#4C72B0", label="60 cấu hình")
    for name, color, label in ((OLD, "#777777", "gốc floor30/open3"),
                               (NEW, "#C44E52", "chốt floor20/open2")):
        row = by_cfg[(condition, name)]
        x, y = float(row["false_blobs_per_img"]), float(row["recall"])
        ax.scatter(x, y, s=130, color=color, edgecolor="black", zorder=5, label=label)
        ax.annotate(name.split("_single")[0], (x, y), textcoords="offset points",
                    xytext=(6, 6), fontsize=8)
    ax.axhline(edp.KPI["recall"], color="#4C72B0", linestyle="--", linewidth=1)
    ax.axvline(edp.KPI["false_blobs"], color="#C44E52", linestyle="--", linewidth=1)
    ax.set_title(f"Điều kiện: {condition}")
    ax.set_xlabel("đốm giả / ảnh (thấp hơn tốt hơn)")
    ax.grid(alpha=0.25)
axes[0].set_ylabel("recall (cao hơn tốt hơn)")
axes[1].legend(loc="lower right", fontsize=8)
fig.suptitle("Không cấu hình nào nằm trong vùng KPI (recall ≥ 0,97 và đốm giả ≤ 0,05)", fontsize=11)
plt.tight_layout()
plt.show()


## 4. Recall theo từng lớp — chỗ cấu hình cũ yếu nhất

`Spur` và `Spurious_copper` là hai lớp khó: vết rất mảnh, dễ bị opening xoá mất.


In [ ]:
classes = ["Missing_hole", "Mouse_bite", "Open_circuit", "Short", "Spur", "Spurious_copper"]
x = np.arange(len(classes))
width = 0.38

fig, ax = plt.subplots(figsize=(12, 4.4))
old_values = [float(by_cfg[("clean", OLD)][f"recall_{c}"]) for c in classes]
new_values = [float(by_cfg[("clean", NEW)][f"recall_{c}"]) for c in classes]
ax.bar(x - width / 2, old_values, width, label="floor30/open3", color="#9E9E9E")
ax.bar(x + width / 2, new_values, width, label="floor20/open2", color="#C44E52")
for index, (a, b) in enumerate(zip(old_values, new_values)):
    ax.text(index - width / 2, a + 0.004, f"{a:.3f}", ha="center", fontsize=7, rotation=90)
    ax.text(index + width / 2, b + 0.004, f"{b:.3f}", ha="center", fontsize=7, rotation=90)
ax.set_xticks(x)
ax.set_xticklabels(classes, fontsize=9)
ax.set_ylim(0.90, 1.01)
ax.set_ylabel("recall (điều kiện clean)")
ax.set_title("Recall từng lớp: cấu hình mới cứu đúng hai lớp yếu nhất")
ax.legend()
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


## 5. Vì sao recall tăng: 3 cặp từng **không trúng lỗi GT nào**

`pairs_all_missed` đếm những cặp mà **không bbox nào trúng Ground Truth** — không có nghĩa là
"không có bbox". Với `floor30/open3`, cả 3 cặp dưới đây hỏng; với `floor20/open2`, cả 3 đều bắt được.
Khung xanh = Ground Truth, khung đỏ = dự đoán. Ảnh được cắt quanh vùng lỗi để nhìn rõ.


In [ ]:
def crop_around(image, boxes, margin: int = 180, max_side: int = 760):
    if not boxes:
        return image
    xs = [b[0] for b in boxes]
    ys = [b[1] for b in boxes]
    xe = [b[0] + b[2] for b in boxes]
    ye = [b[1] + b[3] for b in boxes]
    x0 = max(0, min(xs) - margin)
    y0 = max(0, min(ys) - margin)
    x1 = min(image.shape[1], max(xe) + margin)
    y1 = min(image.shape[0], max(ye) + margin)
    crop = image[y0:y1, x0:x1]
    longest = max(crop.shape[:2])
    if longest > max_side:
        scale = max_side / longest
        crop = cv2.resize(crop, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    return crop


CFG_OLD = (30, 3, 0.0, 20)
CFG_NEW = (20, 2, 0.0, 20)
ns = argparse.Namespace(seed=0, warp_angle=3.0, warp_shift=40.0, warp_scale=0.03)
pair_by_id = {p["pair_id"]: p for p in find_pairs(str(PROJECT_ROOT / "data"))}
demo_results = {}

for pair_id in edp.FOCUS_PAIRS:
    pair = pair_by_id[pair_id]
    template, aligned, diff, valid, info = edp.prepare(pair, "clean", ns)
    mask_old, boxes_old = edp.detect(diff, valid, CFG_OLD)
    mask_new, boxes_new = edp.detect(diff, valid, CFG_NEW)
    gt_boxes = read_labels(pair["label_path"])
    counts_old = detection_counts(boxes_old, gt_boxes)
    counts_new = detection_counts(boxes_new, gt_boxes)
    demo_results[pair_id] = {"gt": len(gt_boxes), "old": counts_old, "new": counts_new}

    old_view = draw_results(aligned, boxes_old)
    new_view = draw_results(aligned, boxes_new)
    for (bx, by, bw, bh) in gt_boxes:
        cv2.rectangle(old_view, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)
        cv2.rectangle(new_view, (bx, by), (bx + bw, by + bh), (0, 255, 0), 2)

    panels = [
        (crop_around(diff, gt_boxes), "bản đồ vi sai"),
        (crop_around(old_view, gt_boxes), f"floor30/open3: trúng {counts_old['gt_hit']}/{len(gt_boxes)}"),
        (crop_around(new_view, gt_boxes), f"floor20/open2: trúng {counts_new['gt_hit']}/{len(gt_boxes)}"),
        (crop_around(mask_new, gt_boxes), "mask floor20/open2"),
    ]
    fig, axes = plt.subplots(1, 4, figsize=(17, 3.4))
    for ax, (image, title) in zip(axes, panels):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(to_display(image, max_width=520))
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    fig.suptitle(f"{pair_id} — GT: {pair['class_name']} ({len(gt_boxes)} lỗi, khung xanh)", fontsize=11)
    plt.tight_layout()
    plt.show()


In [ ]:
print(f"{'pair':<24}{'GT':>4}{'box':>6}{'trúng':>7}{'giả':>6}   |"
      f"{'box':>6}{'trúng':>7}{'giả':>6}")
print(f"{'':<24}{'':>4}{'--- floor30/open3 ---':>19}   |{'--- floor20/open2 ---':>19}")
for pair_id, data in demo_results.items():
    old, new = data["old"], data["new"]
    print(f"{pair_id:<24}{data['gt']:>4}"
          f"{old['n_pred']:>6}{old['gt_hit']:>7}{old['false_blobs']:>6}   |"
          f"{new['n_pred']:>6}{new['gt_hit']:>7}{new['false_blobs']:>6}")


## 6. KPI: nói thật là không đạt

KPI nhóm đặt: `recall ≥ 0,97`, `f1 ≥ 0,975`, `recall lớp yếu ≥ 0,96`, **`đốm giả ≤ 0,05/ảnh`**.
Cột `kpi_pass` trong CSV là kết quả kiểm tra tất cả cùng lúc.


In [ ]:
passing = [r for r in grid if r["kpi_pass"] == "True"]
lowest_fp = min(float(r["false_blobs_per_img"]) for r in grid)
print(f"Cấu hình đạt TẤT CẢ KPI : {len(passing)}/{len(configs)}")
print(f"Đốm giả/ảnh thấp nhất   : {lowest_fp:.4f} (KPI ≤ {edp.KPI['false_blobs']})")
print()
print("→ Nhóm KHÔNG đạt KPI về báo động giả ở bất kỳ cấu hình nào.")
print("→ Việc đổi sang floor20/open2 là để đạt KPI recall và hết cặp sót toàn bộ,")
print("  đổi lại đốm giả tăng nhẹ (resample +0,0087/ảnh).")


## 7. Kiểm chứng và kết luận

`DEFAULT_*` trong `module_preprocess.py` hiện là `floor=20, open=2`. Script
`experiments/verify_defaults.py` chạy lại **693 cặp × 2 điều kiện** bằng đúng default đó
và so từng chỉ số với CSV này (dung sai 2e-5); lần chạy 17/11 khớp 0,00000 ở mọi chỉ số.
Đó là cổng chặn hồi quy — đổi default mà quên chạy lại lưới thì script fail ngay.

```powershell
.\.venv\Scripts\python.exe experiments\verify_defaults.py     # ~11,5 phút
```


---

**Ghi chú khi commit**

```powershell
jupyter nbconvert --clear-output --inplace experiments/notebooks/detection_params.ipynb
```

- Không sửa `exp_detection_params.py`, `detection_params_summary.csv`, `data/` từ notebook.
